# CITE-seq: T/NK normalisation and Treg raw-count export

1. keeps cells whose RNA and protein annotations agree, then T/NK cells with TCR contigs; normalises RNA and exports the
   matrix that `01_aucell_annotation.R` (T cell section) annotates
2. after that annotation: subsets the AUCell Tregs and exports their raw counts (`tcell/notstrict_treg_raw_counts.csv`), the
   CITE-seq query of the Treg label transfer in `cosmx_cd/09_treg_label_transfer.R`

The Treg TCR repertoire analysis is in `05_treg_tcr_repertoire.py`.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import anndata as ad
import scipy.stats as stats
from scipy.stats import gmean, chi2_contingency
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR  = "/path/to/cite-seq/output"
TCELL_DIR = f"{DATA_DIR}/tcell"

## 1. Load data and quality filter

In [ ]:
rna_counts = pd.read_csv(f"{DATA_DIR}/RNA_raw_counts.csv")
anno       = pd.read_csv(f"{DATA_DIR}/annotation_table.csv")

# Keep cells where RNA and protein annotations agree
good_cell = [
    cell for cell, rna_ann, prot_ann
    in zip(anno['Unnamed: 0'], anno['RNA_annotation'], anno['Protein_annotation'])
    if rna_ann == prot_ann
]
print(f"Quality-filtered cells: {len(good_cell):,} / {len(anno):,}")

good_anno = anno[anno['Unnamed: 0'].isin(good_cell)]
good_rna  = rna_counts[rna_counts['Unnamed: 0'].isin(good_cell)]

## 2. Extract T/NK cells with TCR and normalize RNA

In [ ]:
# Load TCR data and restrict to quality-filtered T/NK cells
tcr = pd.read_csv(f"{DATA_DIR}/TCR_merged.csv")
tcr_good = tcr[tcr['barcode'].isin(good_cell)]
tcr_good = tcr_good.merge(good_anno, left_on='barcode', right_on='Unnamed: 0', how='left')
tcr_good = tcr_good[tcr_good['Protein_annotation'] == 'tnk']

# RNA counts for T/NK cells
t_counts = good_rna[good_rna['Unnamed: 0'].isin(tcr_good['barcode'])]
t_counts.index = t_counts['Unnamed: 0'].tolist()
t_counts = t_counts.drop('Unnamed: 0', axis=1)

# Build AnnData and normalize
t_adata = ad.AnnData(t_counts.copy())
t_adata.layers['counts'] = t_counts.values
sc.pp.normalize_total(t_adata)
sc.pp.log1p(t_adata)

# Export normalized counts for AUCell T cell annotation in 01_aucell_annotation.R
t_norm = pd.DataFrame(t_adata.X, index=t_adata.obs_names, columns=t_adata.var_names)
t_norm.to_csv(f"{TCELL_DIR}/good_anno_t_wtcr_norm_counts.csv")
print("Exported T cell normalized counts.")

## 3. Subset AUCell-annotated Tregs and attach dataset metadata

In [ ]:
# Load AUCell T cell labels (from 01_aucell_annotation.R)
aucell_anno = pd.read_csv(f"{TCELL_DIR}/pr_rna_tcr_tcell_aucell.csv")
t_adata.obs['anno'] = aucell_anno['label'].tolist()

# Subset Tregs
treg = t_adata[t_adata.obs['anno'] == 'Treg'].copy()

# Attach dataset metadata for Harmony batch correction
ds = good_anno[good_anno['Unnamed: 0'].isin(treg.obs.index)][['Unnamed: 0', 'dataset']]
treg.obs = treg.obs.merge(ds, how='left', left_index=True, right_on='Unnamed: 0')

print(treg)

## 4. Export Treg raw counts for the pseudotime-bin label transfer

In [ ]:
# Export raw counts → input for the Seurat label transfer in cosmx_cd/09_treg_label_transfer.R
raw_counts = pd.DataFrame(treg.layers['counts'], index=treg.obs.index, columns=treg.var_names)
raw_counts.to_csv(f"{TCELL_DIR}/notstrict_treg_raw_counts.csv")
print("Exported Treg raw counts for pseudotime bin transfer.")